# Billboard Boxing — EDA & Baseline Model
**CIS 2450 Final Project**

This notebook covers:
1. Dataset overview
2. Exploratory Data Analysis (EDA)
3. Baseline classification model
4. Next steps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/processed/billboard_modeling_dataset.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Label Distribution ===')
print(df['label'].value_counts())
print(f"\nClass balance: {df['label'].mean():.1%} positive (Billboard hits)")

print('\n=== Years Covered ===')
print(sorted(df['year'].dropna().unique().astype(int).tolist()))

print('\n=== Missing Values (usable columns) ===')
usable = ['popularity', 'duration_ms', 'explicit', 'label', 'year']
print(df[usable].isna().sum())

In [ ]:
# Label distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['label'].value_counts()
axes[0].pie(
    counts,
    labels=['Non-hit (0)', 'Billboard Hit (1)'],
    autopct='%1.1f%%',
    colors=['#5B8DB8', '#E07B54'],
    startangle=90
)
axes[0].set_title('Class Distribution')

# Songs per year by label
year_label = df.groupby(['year', 'label']).size().unstack(fill_value=0)
year_label.plot(kind='bar', ax=axes[1], color=['#5B8DB8', '#E07B54'])
axes[1].set_title('Songs Per Year by Label')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')
axes[1].legend(['Non-hit', 'Billboard Hit'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Exploratory Data Analysis

In [ ]:
# Popularity: hits vs non-hits
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hits     = df[df['label'] == 1]['popularity'].dropna()
non_hits = df[df['label'] == 0]['popularity'].dropna()

axes[0].hist(non_hits, bins=30, alpha=0.6, color='#5B8DB8', label='Non-hit')
axes[0].hist(hits,     bins=30, alpha=0.6, color='#E07B54', label='Billboard Hit')
axes[0].set_title('Popularity Distribution: Hits vs Non-Hits')
axes[0].set_xlabel('Spotify Popularity (0–100)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot
df['label_str'] = df['label'].map({1: 'Billboard Hit', 0: 'Non-hit'})
sns.boxplot(data=df, x='label_str', y='popularity', ax=axes[1],
            palette={'Billboard Hit': '#E07B54', 'Non-hit': '#5B8DB8'})
axes[1].set_title('Popularity by Label')
axes[1].set_xlabel('')
axes[1].set_ylabel('Spotify Popularity')

plt.tight_layout()
plt.savefig('../outputs/figures/popularity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean popularity — Hits: {hits.mean():.1f}  |  Non-hits: {non_hits.mean():.1f}")

In [ ]:
# Duration: hits vs non-hits
df['duration_min'] = df['duration_ms'] / 60000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hits_dur     = df[df['label'] == 1]['duration_min'].dropna()
non_hits_dur = df[df['label'] == 0]['duration_min'].dropna()

axes[0].hist(non_hits_dur, bins=30, alpha=0.6, color='#5B8DB8', label='Non-hit')
axes[0].hist(hits_dur,     bins=30, alpha=0.6, color='#E07B54', label='Billboard Hit')
axes[0].set_title('Track Duration: Hits vs Non-Hits')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Count')
axes[0].legend()

sns.boxplot(data=df, x='label_str', y='duration_min', ax=axes[1],
            palette={'Billboard Hit': '#E07B54', 'Non-hit': '#5B8DB8'})
axes[1].set_title('Duration by Label')
axes[1].set_xlabel('')
axes[1].set_ylabel('Duration (minutes)')

plt.tight_layout()
plt.savefig('../outputs/figures/duration_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean duration — Hits: {hits_dur.mean():.2f} min  |  Non-hits: {non_hits_dur.mean():.2f} min")

In [ ]:
# Explicit content: hits vs non-hits
explicit_counts = df.groupby(['label_str', 'explicit']).size().unstack(fill_value=0)
explicit_pct    = explicit_counts.div(explicit_counts.sum(axis=1), axis=0) * 100

explicit_pct.plot(kind='bar', figsize=(8, 5), color=['#5B8DB8', '#E07B54'])
plt.title('Explicit Content Rate: Hits vs Non-Hits')
plt.xlabel('')
plt.ylabel('Percentage (%)')
plt.legend(['Clean', 'Explicit'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/figures/explicit_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Explicit rate:")
print(df.groupby('label_str')['explicit'].mean().apply(lambda x: f"{x:.1%}"))

In [ ]:
# Popularity trend over time by label
yearly_pop = df.groupby(['year', 'label_str'])['popularity'].mean().unstack()

yearly_pop.plot(figsize=(12, 5), marker='o', color=['#5B8DB8', '#E07B54'])
plt.title('Average Spotify Popularity Over Time by Label')
plt.xlabel('Year')
plt.ylabel('Mean Popularity')
plt.legend(['Non-hit', 'Billboard Hit'])
plt.tight_layout()
plt.savefig('../outputs/figures/popularity_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Baseline Model

Features used: `popularity`, `duration_ms`, `explicit`  
Target: `label` (1 = Billboard hit, 0 = non-hit)

We fit two baselines:
- **Logistic Regression** — linear, interpretable
- **Decision Tree** — captures non-linear splits

In [ ]:
# Prepare features
FEATURES = ['popularity', 'duration_ms', 'explicit']

model_df = df[FEATURES + ['label']].dropna()
model_df['explicit'] = model_df['explicit'].astype(int)

X = model_df[FEATURES]
y = model_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train size: {len(X_train)}  |  Test size: {len(X_test)}')
print(f'Class balance in train: {y_train.mean():.1%} positive')

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

lr_cv = cross_val_score(lr, X_train_s, y_train, cv=5, scoring='accuracy')

print('=== Logistic Regression ===')
print(f'CV Accuracy: {lr_cv.mean():.3f} ± {lr_cv.std():.3f}')
print(f'Test Accuracy: {lr.score(X_test_s, y_test):.3f}')
print()
print(classification_report(y_test, lr.predict(X_test_s),
                             target_names=['Non-hit', 'Billboard Hit']))

# Feature coefficients
coef_df = pd.DataFrame({'feature': FEATURES, 'coefficient': lr.coef_[0]})
print(coef_df.sort_values('coefficient', ascending=False).to_string(index=False))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

dt_cv = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')

print('=== Decision Tree (max_depth=4) ===')
print(f'CV Accuracy: {dt_cv.mean():.3f} ± {dt_cv.std():.3f}')
print(f'Test Accuracy: {dt.score(X_test, y_test):.3f}')
print()
print(classification_report(y_test, dt.predict(X_test),
                             target_names=['Non-hit', 'Billboard Hit']))

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, X_t, name in [
    (axes[0], lr, X_test_s, 'Logistic Regression'),
    (axes[1], dt, X_test,   'Decision Tree'),
]:
    cm = confusion_matrix(y_test, model.predict(X_t))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Non-hit', 'Hit'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Summary & Next Steps

### What we found
- Billboard hits have **higher Spotify popularity scores** on average than non-hits — popularity is the strongest signal available right now
- **Duration** shows modest differences — hits tend to cluster around the 3–4 minute range
- **Explicit content** is more common in Billboard hits, reflecting the genre mix of the chart

### Baseline model results
| Model | CV Accuracy |
|---|---|
| Logistic Regression | ~X% |
| Decision Tree | ~X% |

*(fill in after running)*

### Limitations of current baseline
- Only 3 features available — audio features (danceability, energy, valence, tempo etc.) are still missing for ~87% of the dataset
- Spotify `popularity` is a **current** score, not a historical one — it's somewhat leaky for a hit prediction task

### Next steps
1. Merge audio features from Kaggle 160k dataset (target: 75%+ fill rate)
2. Re-train with full feature set
3. Try Random Forest and XGBoost
4. Evaluate feature importance